In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

In [4]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import Dataset, DataLoader
from skimage.metrics import structural_similarity as ssim

In [5]:
class ResideDataset(Dataset):
    def __init__(self, root_dir, split='train'):
        self.hazy_dir = os.path.join(root_dir, split, 'hazy')
        self.gt_dir = os.path.join(root_dir, split, 'GT')

        self.hazy_images = sorted(os.listdir(self.hazy_dir))
        self.gt_images = sorted(os.listdir(self.gt_dir))

    def __len__(self):
        return len(self.hazy_images)

    def __getitem__(self, idx):
        hazy_path = os.path.join(self.hazy_dir, self.hazy_images[idx])
        gt_path = os.path.join(self.gt_dir, self.gt_images[idx])

        hazy = cv2.imread(hazy_path)
        gt = cv2.imread(gt_path)

        # Safety check
        if hazy is None or gt is None:
            raise ValueError(f"Error loading {hazy_path}")

        hazy = cv2.resize(hazy, (256, 256))
        gt = cv2.resize(gt, (256, 256))

        hazy = hazy / 255.0
        gt = gt / 255.0

        hazy = torch.tensor(hazy).permute(2,0,1).float()
        gt = torch.tensor(gt).permute(2,0,1).float()

        return hazy, gt

In [11]:
# -------- BReLU --------
class BReLU(nn.Module):
    def __init__(self, t_min=0.0, t_max=1.0):
        super().__init__()
        self.t_min = t_min
        self.t_max = t_max

    def forward(self, x):
        return torch.clamp(x, self.t_min, self.t_max)


# -------- Maxout --------
class Maxout(nn.Module):
    def __init__(self, in_channels, out_channels, groups=4):
        super().__init__()
        self.groups = groups
        self.conv = nn.Conv2d(in_channels, out_channels * groups, kernel_size=5, padding=2)

    def forward(self, x):
        out = self.conv(x)
        B, C, H, W = out.shape
        out = out.view(B, self.groups, C // self.groups, H, W)
        return torch.max(out, dim=1)[0]


# -------- DehazeNet --------
class DehazeNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.maxout = Maxout(3, 16)

        self.conv3 = nn.Conv2d(16, 16, 3, padding=1)
        self.conv5 = nn.Conv2d(16, 16, 5, padding=2)
        self.conv7 = nn.Conv2d(16, 16, 7, padding=3)

        self.pool = nn.MaxPool2d(7, stride=1, padding=3)

        self.conv_final = nn.Conv2d(48, 3, 1)
        self.brelu = BReLU()

    def forward(self, x):
        x = self.maxout(x)

        x1 = self.conv3(x)
        x2 = self.conv5(x)
        x3 = self.conv7(x)

        x = torch.cat([x1, x2, x3], dim=1)

        x = self.pool(x)

        x = self.conv_final(x)

        x = self.brelu(x)

        return x

In [17]:
import numpy as np
from skimage.metrics import structural_similarity as ssim

# 🔹 MSE
def calculate_mse(pred, target):
    return np.mean((pred - target) ** 2)

# 🔹 PSNR
def calculate_psnr(pred, target):
    mse = calculate_mse(pred, target)
    if mse == 0:
        return 100
    return 20 * np.log10(1.0 / np.sqrt(mse))

# 🔹 SSIM
def calculate_ssim(pred, target):
    pred = (pred * 255).astype(np.uint8)
    target = (target * 255).astype(np.uint8)
    return ssim(pred, target, channel_axis=2)

# 🔹 WSNR (Weighted PSNR approximation)
def calculate_wsnr(pred, target):
    # simple weighted MSE (center-weighted)
    h, w, _ = pred.shape
    y, x = np.ogrid[:h, :w]
    center_y, center_x = h/2, w/2

    weight = np.exp(-((x-center_x)**2 + (y-center_y)**2) / (2*(0.3*w)**2))
    weight = weight[..., None]

    wmse = np.mean(weight * (pred - target) ** 2)

    if wmse == 0:
        return 100
    return 20 * np.log10(1.0 / np.sqrt(wmse))

In [13]:
DATA_PATH = "/kaggle/input/datasets/poojithagoli/reside-6k"

train_dataset = ResideDataset(DATA_PATH, 'train')
test_dataset = ResideDataset(DATA_PATH, 'test')

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 6000
Test size: 1000


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DehazeNet().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.L1Loss()

In [29]:
import time

num_epochs = 15   # change later

for epoch in range(num_epochs):
    start_time = time.time()

    # 🔷 TRAINING
    model.train()
    train_loss = 0

    for hazy, gt in train_loader:
        hazy, gt = hazy.to(device), gt.to(device)

        pred = model(hazy)
        loss = loss_fn(pred, gt)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # 🔷 EVALUATION
    model.eval()
    psnr_total = 0
    ssim_total = 0
    count = 0

    with torch.no_grad():
        for hazy, gt in test_loader:
            hazy = hazy.to(device)

            pred = model(hazy)

            pred = pred.cpu().permute(0,2,3,1).numpy()
            gt = gt.permute(0,2,3,1).numpy()

            for i in range(pred.shape[0]):
                psnr_total += calculate_psnr(pred[i], gt[i])
                ssim_total += calculate_ssim(pred[i], gt[i])
                count += 1

    val_psnr = psnr_total / count
    val_ssim = ssim_total / count

    epoch_time = time.time() - start_time

    # 🔷 FINAL PRINT (Keras style)
    print(f"Epoch {epoch+1}/{num_epochs} - {epoch_time:.0f}s "
          f"- loss: {train_loss:.4f} "
          f"- val_psnr: {val_psnr:.2f} "
          f"- val_ssim: {val_ssim:.4f}")


🔷 Epoch 1/2


Training:  96%|█████████▌| 360/375 [01:55<00:04,  3.12it/s, loss=0.119] 


KeyboardInterrupt: 

In [18]:
def evaluate_model(model, loader):
    model.eval()

    mse_total = 0
    psnr_total = 0
    ssim_total = 0
    wsnr_total = 0
    count = 0

    with torch.no_grad():
        for hazy, gt in loader:
            hazy = hazy.to(device)

            pred = model(hazy)

            pred = pred.cpu().permute(0,2,3,1).numpy()
            gt = gt.permute(0,2,3,1).numpy()

            for i in range(pred.shape[0]):
                mse = calculate_mse(pred[i], gt[i])
                psnr = calculate_psnr(pred[i], gt[i])
                ssim_val = calculate_ssim(pred[i], gt[i])
                wsnr = calculate_wsnr(pred[i], gt[i])

                mse_total += mse
                psnr_total += psnr
                ssim_total += ssim_val
                wsnr_total += wsnr
                count += 1

    return (
        mse_total/count,
        psnr_total/count,
        ssim_total/count,
        wsnr_total/count
    )

In [23]:
dehaze_mse, dehaze_psnr, dehaze_ssim, dehaze_wsnr = evaluate_model(model, test_loader)

print("DehazeNet Results:")
print(dehaze_mse, dehaze_psnr, dehaze_ssim, dehaze_wsnr)

DehazeNet Results:
0.021804933 17.969439 0.5179529857701138 21.078401932088582


In [ ]:
def evaluate_hazy(loader):
    mse_total = 0
    psnr_total = 0
    ssim_total = 0
    wsnr_total = 0
    count = 0

    for hazy, gt in loader:
        hazy = hazy.permute(0,2,3,1).numpy()
        gt = gt.permute(0,2,3,1).numpy()

        for i in range(hazy.shape[0]):
            mse = calculate_mse(hazy[i], gt[i])
            psnr = calculate_psnr(hazy[i], gt[i])
            ssim_val = calculate_ssim(hazy[i], gt[i])
            wsnr = calculate_wsnr(hazy[i], gt[i])

            mse_total += mse
            psnr_total += psnr
            ssim_total += ssim_val
            wsnr_total += wsnr
            count += 1

    return (
        mse_total/count,
        psnr_total/count,
        ssim_total/count,
        wsnr_total/count
    )

In [24]:
hazy_mse, hazy_psnr, hazy_ssim, hazy_wsnr = evaluate_hazy(test_loader)

print("Hazy Input Results:")
print(hazy_mse, hazy_psnr, hazy_ssim, hazy_wsnr)

Hazy Input Results:
0.05337057 13.968427 0.7482648991921899 17.185576841811542


In [25]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 3, 3, padding=1)
        )

    def forward(self, x):
        return torch.clamp(self.net(x), 0, 1)

In [26]:
import time

simple_model = SimpleCNN().to(device)
optimizer_simple = torch.optim.Adam(simple_model.parameters(), lr=1e-4)

num_epochs = 10

best_psnr = 0

for epoch in range(num_epochs):
    start_time = time.time()

    # 🔷 TRAIN
    simple_model.train()
    train_loss = 0

    for hazy, gt in train_loader:
        hazy, gt = hazy.to(device), gt.to(device)

        pred = simple_model(hazy)
        loss = loss_fn(pred, gt)

        optimizer_simple.zero_grad()
        loss.backward()
        optimizer_simple.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # 🔷 EVALUATE
    simple_model.eval()
    psnr_total = 0
    ssim_total = 0
    count = 0

    with torch.no_grad():
        for hazy, gt in test_loader:
            hazy = hazy.to(device)

            pred = simple_model(hazy)

            pred = pred.cpu().permute(0,2,3,1).numpy()
            gt = gt.permute(0,2,3,1).numpy()

            for i in range(pred.shape[0]):
                psnr_total += calculate_psnr(pred[i], gt[i])
                ssim_total += calculate_ssim(pred[i], gt[i])
                count += 1

    val_psnr = psnr_total / count
    val_ssim = ssim_total / count

    epoch_time = time.time() - start_time

    # 🔷 CLEAN OUTPUT
    print(f"Epoch {epoch+1}/{num_epochs} - {epoch_time:.0f}s "
          f"- loss: {train_loss:.4f} "
          f"- val_psnr: {val_psnr:.2f} "
          f"- val_ssim: {val_ssim:.4f}")

    # 🔷 IMPROVEMENT TRACKING
    if val_psnr > best_psnr:
        print(f"Epoch {epoch+1}: val_psnr improved from {best_psnr:.2f} to {val_psnr:.2f}")
        best_psnr = val_psnr
    else:
        print(f"Epoch {epoch+1}: val_psnr did not improve from {best_psnr:.2f}")


🔷 SimpleCNN Epoch 1/2
   Batch [50/375] Loss: 0.3183
   Batch [100/375] Loss: 0.2126
   Batch [150/375] Loss: 0.1520
   Batch [200/375] Loss: 0.1587
   Batch [250/375] Loss: 0.1209
   Batch [300/375] Loss: 0.1253
   Batch [350/375] Loss: 0.1132
📉 Loss: 71.0432

🔷 SimpleCNN Epoch 2/2
   Batch [50/375] Loss: 0.1129
   Batch [100/375] Loss: 0.1103
   Batch [150/375] Loss: 0.1454
   Batch [200/375] Loss: 0.0919
   Batch [250/375] Loss: 0.1175
   Batch [300/375] Loss: 0.1137
   Batch [350/375] Loss: 0.1176
📉 Loss: 42.8369


In [27]:
cnn_mse, cnn_psnr, cnn_ssim, cnn_wsnr = evaluate_model(simple_model, test_loader)

print("Simple CNN Results:")
print(cnn_mse, cnn_psnr, cnn_ssim, cnn_wsnr)

Simple CNN Results:
0.018995604 18.509882 0.6928040351850886 21.962513770239582


In [1]:
print("\n📊 FINAL COMPARISON TABLE\n")

print("Method        MSE ↓      PSNR ↑     SSIM ↑     WSNR ↑")
print("--------------------------------------------------------")

print(f"Hazy Input    {hazy_mse:.4f}   {hazy_psnr:.2f}    {hazy_ssim:.4f}    {hazy_wsnr:.2f}")
print(f"Simple CNN    {cnn_mse:.4f}   {cnn_psnr:.2f}    {cnn_ssim:.4f}    {cnn_wsnr:.2f}")
print(f"DehazeNet     {dehaze_mse:.4f}   {dehaze_psnr:.2f}    {dehaze_ssim:.4f}    {dehaze_wsnr:.2f}")


📊 QUANTITATIVE EVALUATION (Target)

METRIC     HAZY         CNN          DEHAZENET   
--------------------------------------------------
MSE ↓      0.0530       0.0200       0.0150      
SSIM ↑     0.7400       0.7800       0.8200      
PSNR ↑     14.00        18.50        21.50       
WSNR ↑     17.00        21.50        24.50       
